In [28]:
import os
os.environ["OPENAI_API_KEY"] ='provide your api key'

In [2]:
# llm.py
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def ask_llm(prompt):
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return res.choices[0].message.content

In [14]:
# current_agent.py
#from llm import ask_llm

def planner(goal):
    tasks = ask_llm(f"Break into 5 steps:\n{goal}")
    #print(tasks)
    return [t for t in tasks.split("### Step") if t.strip()]

def delegator(task):
    t = task.lower()
    if "code" in t or "python" in t:
        return "code"
    elif "write" in t or "blog" in t:
        return "writer"
    return "general"

def execute(agent, task):
    return ask_llm(f"{agent} agent: {task}")

def verify(task, output):
    res = ask_llm(f"Is this acceptable? YES/NO\n{output}")
    return "YES" in res.upper()

def run_current(goal):
    print("\n=== CURRENT AGENT ===")
    tasks = planner(goal)
    print(len(tasks))

    for task in tasks:
        print(f"\nTask: {task}")

        agent = delegator(task)
        print(f"Assigned to: {agent}")

        output = execute(agent, task)
        print(f"Output: {output[:150]}...")

        if verify(task, output):
            print("✅ Accepted")
        else:
            print("❌ Failed (but no real recovery)")

In [15]:
goal = "Write a blog on AI trends and include a Python example"

run_current(goal)


=== CURRENT AGENT ===
6

Task: Sure! Here’s a structured approach to writing a blog on AI trends, along with a Python example. 


Assigned to: code
Output: ### Title: Top AI Trends to Watch in [Year]

#### Introduction
In recent years, artificial intelligence (AI) has played a pivotal role in revolutioniz...
✅ Accepted

Task:  1: Introduction to AI Trends

Begin your blog with a brief introduction to the topic of AI trends. Explain why keeping track of these trends is important for both professionals in the field and businesses looking to innovate.

**Example Intro:**
"Artificial Intelligence (AI) is rapidly evolving, influencing various sectors from healthcare to finance. Understanding the current trends in AI not only helps professionals stay ahead of the curve but also provides businesses with insights to leverage these technologies for competitive advantage. In this blog, we’ll explore the most significant AI trends and showcase a practical Python example to illustrate one of thes

In [25]:
# intelligent_agent.py
import random

# --- Agent metadata ---
agents = {
    "writer": {"success": 0.9, "cost": 1},
    "code": {"success": 0.7, "cost": 2},
    "general": {"success": 0.6, "cost": 0.5}
}

# --- Planner ---
def planner(goal):
    tasks = ask_llm(f"Break into 4 steps:\n{goal}")
    #print(tasks)
    return [t for t in tasks.split("### Step") if t.strip()]

# --- Confidence estimation ---
def estimate_confidence(task):
    res = ask_llm(f"Rate confidence (0-1) for completing:\n{task}")
    try:
        return float(res.strip())
    except:
        return 0.5

# --- Smart delegator ---
def choose_agent(task):
    confidence = estimate_confidence(task)

    best_agent = None
    best_score = -999

    for name, data in agents.items():
        score = (data["success"] * confidence) - data["cost"]

        if score > best_score:
            best_score = score
            best_agent = name

    return best_agent, confidence

# --- Execute ---
def execute(agent, task):
    return ask_llm(f"{agent} agent: {task}")

# --- Verification ---
def verify(task, output):
    res = ask_llm(f"""
    Task: {task}
    Output: {output}

    Is this high quality? YES or NO.
    """)
    return "YES" in res.upper()

# --- Learning ---
def update(agent, success):
    if success:
        agents[agent]["success"] += 0.05
    else:
        agents[agent]["success"] -= 0.1

# --- Main ---
def run_intelligent(goal):
    print("\n=== INTELLIGENT DELEGATION AGENT ===")

    tasks = planner(goal)
    print(len(tasks))
    for task in tasks:
        print(f"\nTask: {task}")

        agent, confidence = choose_agent(task)
        print(f"Chosen agent: {agent} (confidence: {confidence:.2f})")

        output = execute(agent, task)
        print(f"Output: {output[:150]}...")

        if verify(task, output):
            print("✅ Verified")
            update(agent, True)

        else:
            print("❌ Failed → Re-delegating")

            # try better agent (simulate upgrade)
            fallback = "writer" if agent != "writer" else "code"

            output = execute(fallback, task)
            print(f"Retry with {fallback}: {output[:150]}...")

            if verify(task, output):
                print("✅ Recovered")
                update(fallback, True)
            else:
                print("❌ Escalation needed")
                update(agent, False)

In [26]:
goal = "Write a blog on AI trends and include a Python example"

run_intelligent(goal)


=== INTELLIGENT DELEGATION AGENT ===
5

Task: Sure! Here’s a structured outline for a blog post on AI trends, broken down into four main steps, along with a Python example to illustrate one of the trends.


Chosen agent: general (confidence: 0.50)
Output: Here's a structured outline for a blog post on AI trends, featuring four main steps and including a Python example to illustrate one of the trends.

-...
✅ Verified

Task:  1: Introduction to AI Trends

**Title: Emerging Trends in Artificial Intelligence: Shaping the Future**

In recent years, Artificial Intelligence (AI) has rapidly evolved, transforming industries and enhancing everyday life. From healthcare to finance, AI is reshaping how we approach problem-solving and decision-making. In this blog, we will explore some of the most significant trends in AI today, including AI ethics, automation, natural language processing (NLP), and generative AI. Additionally, I will showcase a brief Python example that demonstrates a simple AI